# Inventory Diff and Reference Generation

Runs inventory change detection and per-object reference generation. 

In [10]:
import sys
import warnings
from pathlib import Path
from datetime import datetime, UTC
from dask.distributed import Client
from uuid import uuid4
import pandas as pd

from utils.config_utils import (
    check_runtime_readiness,
    load_pipeline_config,
    resolve_secrets,
)
from pipeline.inventory import build_inventory_snapshot_and_diff
from pipeline import generate_json as gjson

from pipeline.json_sanitizer import sanitize_generated_references

print("Imports OK")

Imports OK


In [11]:
check_runtime_readiness()
kp = load_pipeline_config("configs/config.yaml")
ACCESS_KEY, SECRET_KEY = resolve_secrets(kp)

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

## S3 Inventory building

In [12]:
ledger = build_inventory_snapshot_and_diff(
    kp=kp,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
)

print("Inventory summary:", ledger["summary"])

inventory_diff = ledger["diff"]
inventory_objects = ledger["current_objects"]
pending_ledger = ledger["next_ledger"]

Inventory summary: {'scanned': 54, 'new': 1, 'changed': 53, 'deleted': 1, 'unchanged': 0}


## Dask client init

In [8]:
exec_cfg = kp.get("execution", {})
raw_workers = exec_cfg.get("max_workers", "auto")
workers_number = gjson._resolve_workers(raw_workers)
worker_mem_limit = exec_cfg.get("memory_limit_gb", "1GB")

client = Client(
    n_workers=workers_number, threads_per_worker=1, memory_limit=worker_mem_limit
)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 3.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:64927,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:64950,Total threads: 1
Dashboard: http://127.0.0.1:64951/status,Memory: 0.93 GiB
Nanny: tcp://127.0.0.1:64930,


## Reference Generation (Full Run)

Processes all new/changed objects in parallel via Dask.

Not using concurrency (threads_per_worker> 1) because HDF5 doesn't like it

In [9]:
full_start_time = datetime.now(UTC)

reference_generation = gjson.parallel_dask_ref_generation(
    client=client,
    kp=kp,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    inventory_diff=inventory_diff,
    current_objects=inventory_objects,
)

full_end_time = datetime.now(UTC)
gen_duration = str(full_end_time - full_start_time).split(".")[0]

print("Reference generation summary:", reference_generation["summary"])
print("Full-run duration:", gen_duration)

Reference generation summary: {'scanned': 54, 'changed_or_new': 54, 'generated': 54, 'skipped': 0, 'failed': 0, 'deleted_refs_removed': 1, 'deleted_refs_missing': 0, 'timestamp': '2026-07-21T05:19:39.688059+00:00'}
Full-run duration: 0:12:30


## Sanitize generated JSON references to remove _FillValue: NaN

The underlying NetCDF can have CF _FillValue: np.float64(nan), but JSON must be strictly-typed to be browser valid. 

It must use base64 representation of NaN which is "_FillValue": "AAAAAAAA+H8=", when this is decoded it becomes float64(nan)

**Note**: In Zarr V2 .zarray "fill_value": "NaN" is valid and expected.

In [13]:
generated_source_keys= [
    result["source_key"] 
    for result in reference_generation["results"]
    ]

sanitize_summary = sanitize_generated_references(staging_volume_path = kp["output"]["staging_volume_path"], source_keys = generated_source_keys)

print("Sanitising JSON references result: ", sanitize_summary)

Sanitising JSON references result:  {'checked_refs': 54, 'sanitized_zattrs_instances': 2}


## Save inventory snapshot

After successful `gjson.parallel_dask_ref_generation()`, commit ledger to `acacia_refs_staging/_state/inventory_ledger.json`.

Additonally, saves full run duration (from starting parallel_dask to saving ledger) in `.runtime_logs/full_run.csv`. Will append as new row if already exists

In [14]:
from uuid import uuid4

runtime_log_path = Path(".runtime_logs/full_run.csv")
runtime_log_path.parent.mkdir(exist_ok=True)

commit_start = datetime.now(UTC)
if reference_generation["summary"]["failed"] == 0:
    gjson.save_ledger_after_success(
        ledger_path=kp["output"]["ledger_path"],
        next_ledger=pending_ledger,
        generation_summary=reference_generation["summary"],
    )
    commit_status = "committed"
    print("Ledger committed:", kp["output"]["ledger_path"])
else:
    commit_status = "blocked_due_to_failures"
    print("Ledger NOT committed due to generation failures.")

commit_end = datetime.now(UTC)
commit_ledger_time = str(commit_end - commit_start).split(".")[0]

runtime_log_row = {
    "run_id": str(uuid4()),
    "start_time": full_start_time,
    "ended_at": commit_end,
    "gen_duration": gen_duration,
    "commit_ledger_time": commit_ledger_time,
    "scanned": int(reference_generation["summary"].get("scanned", 0)),
    "changed_or_new": int(reference_generation["summary"].get("changed_or_new", 0)),
    "generated": int(reference_generation["summary"].get("generated", 0)),
    "failed": int(reference_generation["summary"].get("failed", 0)),
    "commit_status": commit_status,
}

df = pd.DataFrame([runtime_log_row])
header = not runtime_log_path.exists()
df.to_csv(runtime_log_path, mode="a", index=False, header=header)

print(f"Runtime log appended to: {runtime_log_path}")
print("Full-run duration:", gen_duration)
print("Commit ledger time:", commit_ledger_time)

Ledger committed: acacia_refs_staging/_state/inventory_ledger.json
Runtime log appended to: .runtime_logs\full_run.csv
Full-run duration: 0:12:30
Commit ledger time: 0:00:00


In [15]:
client.close()